# Sentiment Analyzer for Product Reviews

**EDP Project — Batch 8**

This notebook demonstrates text cleaning, TF-IDF feature extraction, Logistic Regression classification, model evaluation, error analysis, and custom review prediction.

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


## 1. Load the Product Review Dataset

In [ ]:
df = pd.read_csv('data/reviews.csv')
df.head()

In [ ]:
print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['sentiment'].value_counts())

## 2. Text Cleaning

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\\S+|www\\S+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']].head()

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment'],
    test_size=0.25, random_state=42, stratify=df['sentiment']
)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 4. TF-IDF + Logistic Regression

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Model training completed.')

## 5. Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, pred)
print(f'Accuracy: {accuracy:.2f}\n')
print(classification_report(y_test, pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, pred, labels=['negative', 'positive'])
print(cm)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['negative', 'positive'])
disp.plot()
plt.title('Confusion Matrix')
plt.show()

## 6. Error Analysis

In [ ]:
errors = pd.DataFrame({
    'review': X_test,
    'actual': y_test,
    'predicted': pred
})
errors = errors[errors['actual'] != errors['predicted']]

if errors.empty:
    print('No errors on this demonstration test split.')
else:
    display(errors)

## 7. Limitations

- The demonstration dataset is small.
- The model supports only positive and negative sentiment.
- TF-IDF may not understand sarcasm or complex context.
- Performance depends on the size and quality of real-world training data.

## 8. Test a Custom Review

In [ ]:
def predict_review(review):
    cleaned = clean_text(review)
    return model.predict([cleaned])[0]

test_review = 'The battery is excellent and the phone is very easy to use.'
print('Review:', test_review)
print('Predicted sentiment:', predict_review(test_review))

## Conclusion

The project demonstrates a complete basic NLP sentiment-analysis workflow: preprocessing, TF-IDF feature extraction, Logistic Regression training, evaluation, error analysis, and prediction on new product reviews.